# Cultist Wars policy pilot
Upload cultist-nn-41710886.zip as input. Select GPU T4 x2. Run cells in order. Return final report and results ZIP. No submission or C integration performed.

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile

work = Path('/kaggle/working/cultist-nn-41710886')
inputs = Path('/kaggle/input')
candidates = [p.parent for p in inputs.rglob('train.py') if (p.parent / 'manifest.json').exists()]
if candidates:
    shutil.copytree(candidates[0], work, dirs_exist_ok=True)
else:
    archives = list(inputs.rglob('cultist-nn-41710886.zip'))
    assert archives, 'Add the supplied training bundle as a Kaggle input first.'
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall('/kaggle/working')
# Dataset upload may expand nested gzip files. Verify original payload before restoring.
import gzip
payload_sha256 = "8de82a9a36ea84159f137fb82ac27a820b469ffe63b5745c95293054e5a5e62c"
recovered = False
if not (work / 'games.jsonl.gz').exists():
    plain = list(work.rglob('games.jsonl'))
    assert plain, 'Replay data missing: expand dataset tree and report filenames.'
    payload = plain[0].read_bytes()
    assert hashlib.sha256(payload).hexdigest() == payload_sha256, 'Replay payload mismatch'
    (work / 'games.jsonl.gz').write_bytes(gzip.compress(payload, mtime=0))
    recovered = True
manifest = json.loads((work / 'manifest.json').read_text())
for name, digest in manifest.items():
    if name == 'games.jsonl.gz':
        assert hashlib.sha256(gzip.decompress((work / name).read_bytes())).hexdigest() == payload_sha256
        continue  # Verify payload; gzip headers may differ.
    assert hashlib.sha256((work / name).read_bytes()).hexdigest() == digest, name
print('Bundle integrity: PASS')
subprocess.run([sys.executable, str(work / 'prepare.py')], check=True)


In [ ]:
subprocess.run([
    sys.executable, str(work / 'train.py'),
    '--device', 'cuda:0', '--minutes', '12', '--epochs', '80'
], check=True)


In [ ]:
from IPython.display import FileLink, display
result = shutil.make_archive('/kaggle/working/cultist-nn-results', 'zip', work / 'output')
print('=== PASTE BACK START ===')
print((work / 'output/report.json').read_text())
print('=== PASTE BACK END ===')
display(FileLink(result))
